# F-02: Stocking Density + Pruned Management + Leave-One-Group-In (RFm)

Follow-up to F-01. Three things:
1. **Stocking density** features (LSU/ha + per-species/ha) from NWFP fenced areas (Design_Develop Appendix D, D-29).
2. **Pruned management** (tower-specific cut + manure recency only) — F-01's 12-col management set overfit.
3. **Leave-one-group-in** ablation (BASE vs BASE+each single group) — clean per-group attribution (F-01 was cumulative, so P2's damage masked later groups).

**Key caveat (honest):** Catchments 4 and 9 are *both 7.75 ha*, so density = LSU/7.75 is a constant rescale of LSU. RFm is invariant to monotonic rescaling of a single feature → for single-tower (or T4+T9-pooled) models, **count and density give identical R²**. Density only adds information when catchments of *different* area share one model — demonstrated in the pooled section by adding **Tower 2's 2018 grassland data** (Catchment 2 = 6.65 ha).

BASE = 03b R-02-CO2 RFm (driver_m + gap-filled FCO2). Harness = R-02 gaps (vs/s/m/l/m1, 25%, 5 reps).

In [1]:
from pathlib import Path
import datetime
import numpy as np
import pandas as pd
from sklearn.ensemble import RandomForestRegressor
from sklearn.impute import SimpleImputer
from sklearn.metrics import r2_score, mean_squared_error, mean_absolute_error

HOURLY  = Path("../../data/Hourly")
RESULTS = Path("../../results")

PLAUS_LOW, PLAUS_HIGH = -500, 3000
N_REPS, MASK_FRAC = 5, 0.25
SCENARIOS = {"vs":1, "s":4, "m":32, "l":288, "m1":"mixed"}
LSU = {"cattle":1.0, "sheep":0.1, "lamb":0.05}
AUX = ["_hour_sin","_hour_cos","_doy_sin","_doy_cos"]

# Total fenced area (ha) per catchment - Design_Develop Appendix D (D-29)
CATCHMENT_AREA_HA = {1:4.81, 2:6.65, 3:6.62, 4:7.75, 5:6.47, 6:3.86, 7:2.60, 8:7.02,
                     9:7.75, 10:1.82, 11:1.76, 12:1.78, 13:1.75, 14:1.72, 15:1.54}

## 1  Tower configs (2, 4, 9)

In [2]:
C4 = "Catchment 4 After  2013/08/13"
def cfg(t, cat, catstr):
    return {
        "tower":t, "catchment":cat, "area_ha":CATCHMENT_AREA_HA[cat],
        "target":f"FCH4_1_1_1 [Tower {t}]", "ssitc":f"FCH4_SSITC_TEST_1_1_1 [Tower {t}]",
        "sw":f"SWIN_1_1_1 [Tower {t}]", "ta":f"TA_0_0_1 [Tower {t}]", "vpd":f"VPD_0_0_1 [Tower {t}]",
        "ppfd":f"PPFD_1_1_1 [Tower {t}]", "ustar":f"USTAR_0_0_1 [Tower {t}]", "ws":f"WS_0_0_1 [Tower {t}]",
        "rn":f"RN_1_1_1 [Tower {t}]", "precip":f"Precipitation (mm) [{catstr}]", "ts":"TS_1_1_1 [Tower 9]",
        "swc":f"Soil Moisture @ 10cm Depth (%) [{catstr}]", "shf":f"SHF_1_1_1 [Tower {t}]",
        "fc":f"FC_1_1_1 [Tower {t}]", "wd":f"WD_0_0_1 [Tower {t}]",
        "soiltemp15":f"Soil Temperature @ 15cm Depth (oC) [{catstr}]",
        "livestock":{sp:f"{sp}_{catstr}" for sp in LSU},
        "chem":[f"Nitrite and Nitrate (mg/l) [{catstr}]", f"Ammonium (mg/l) [{catstr}]",
                f"Dissolved Oxygen (%) [{catstr}]", f"Conductivity (uS/cm) [{catstr}]"],
        "mgmt_scope":f"t{t}",
        "train_yrs":[2018] if t==2 else list(range(2018,2022)),
        "test_yrs":[2019] if t==2 else list(range(2022,2024)),
    }
TOWER_CONFIGS = {4: cfg(4,4,C4), 9: cfg(9,9,"Catchment 9"), 2: cfg(2,2,"Catchment 2")}

## 2  Load data

In [3]:
df_raw = pd.read_csv(HOURLY/"consolidated_hourly.csv", low_memory=False)
df_raw["Datetime"]=pd.to_datetime(df_raw["Datetime"], format="mixed"); df_raw=df_raw.set_index("Datetime")
fco2 = pd.read_csv(HOURLY/"fco2_gapfilled.csv", low_memory=False)
fco2["Datetime"]=pd.to_datetime(fco2["Datetime"], format="mixed"); fco2=fco2.set_index("Datetime")
for _t in [2,4,9]:
    c=f"FC_1_1_1 [Tower {_t}]"
    if c in df_raw.columns: df_raw[c]=fco2[f"FC_gapfilled [Tower {_t}]"]
mgmt = pd.read_csv(HOURLY/"management_features.csv", low_memory=False)
mgmt["Datetime"]=pd.to_datetime(mgmt["Datetime"], format="mixed"); mgmt=mgmt.set_index("Datetime")
df_raw = df_raw.join(mgmt, how="left")
print(f"Loaded {df_raw.shape}")

Loaded (70153, 467)


## 3  Functions

In [4]:
def insert_calendar_gaps(df_qc, target, test_yrs, gap_hours, n_reps=N_REPS, seed=0):
    tm=df_qc.index.year.isin(test_yrs); tt=df_qc.index[tm]
    va=df_qc.loc[tm,target].notna().values; n=len(tt); tn=max(1,int(va.sum()*MASK_FRAC))
    rb=np.random.default_rng(seed); reps=[]
    for _ in range(n_reps):
        rng=np.random.default_rng(int(rb.integers(0,2**31))); occ=np.zeros(n,bool); m=0
        for sp in rng.permutation(n):
            if m>=tn: break
            gh=int(rng.choice([1,4,32,288])) if gap_hours=="mixed" else gap_hours
            ep=min(int(sp)+gh,n)
            if occ[sp:ep].any(): continue
            occ[sp:ep]=True; m+=int(va[sp:ep].sum())
        reps.append(tt[occ & va])
    return reps

def add_time(d):
    h,doy=d.index.hour,d.index.dayofyear
    d["_hour_sin"]=np.sin(2*np.pi*h/24); d["_hour_cos"]=np.cos(2*np.pi*h/24)
    d["_doy_sin"]=np.sin(2*np.pi*doy/365); d["_doy_cos"]=np.cos(2*np.pi*doy/365)

In [5]:
def build_features(df, c):
    """Per-tower feature frame + leave-one-group-in groups dict."""
    d=df.copy(); add_time(d)
    BASE=[c["sw"],c["ta"],c["vpd"],c["ppfd"],c["ustar"],c["ws"],c["rn"],c["precip"],c["ts"],c["swc"],c["shf"]]+AUX+[c["fc"]]
    liv=c["livestock"]; area=c["area_ha"]
    d["_lsu"]=sum(d[liv[s]].fillna(0)*w for s,w in LSU.items())
    d["_graze"]=(sum(d[liv[s]].fillna(0) for s in LSU)>0).astype(float)
    d["_cattle_lag24"]=d[liv["cattle"]].shift(24); d["_lsu_lag168"]=d["_lsu"].shift(168)
    # stocking density (D-29): head or LSU per hectare of the catchment
    d["_lsu_dens"]=d["_lsu"]/area
    for s in LSU: d[f"_{s}_dens"]=d[liv[s]].fillna(0)/area
    d["_wd_sin"]=np.sin(np.deg2rad(d[c["wd"]])); d["_wd_cos"]=np.cos(np.deg2rad(d[c["wd"]]))
    d["_precip_7d"]=d[c["precip"]].rolling(168,min_periods=1).sum()
    d["_precip_30d"]=d[c["precip"]].rolling(720,min_periods=1).sum()
    d["_swc_ante7d"]=d[c["swc"]].rolling(168,min_periods=1).mean()
    d["_prod_proxy"]=d[c["sw"]]*d[c["ta"]]
    sc=c["mgmt_scope"]
    groups={
        "livestock_count":[liv["cattle"],liv["sheep"],liv["lamb"],"_lsu","_graze","_cattle_lag24","_lsu_lag168"],
        "livestock_density":["_lsu_dens","_cattle_dens","_sheep_dens","_lamb_dens"],
        "mgmt_pruned":[f"mgmt_{sc}_cut_recency", f"mgmt_{sc}_manure_recency"],
        "wind":["_wd_sin","_wd_cos"],
        "moisture":["_precip_7d","_precip_30d","_swc_ante7d"],
        "soiltemp_prod":[c["soiltemp15"],"_prod_proxy"],
        "chem":[x for x in c["chem"] if x in d.columns],
    }
    return d, BASE, groups

def fit_eval(d, target, feat, train_idx, y_train, rep_gaps):
    imp=SimpleImputer(strategy="mean"); rf=RandomForestRegressor(n_estimators=500,min_samples_leaf=5,n_jobs=-1,random_state=42)
    rf.fit(imp.fit_transform(d.loc[train_idx,feat].values), y_train)
    rec=[]
    for sc,reps in rep_gaps.items():
        for rep,ts in enumerate(reps):
            if len(ts)<5: continue
            yt=d.loc[ts,target].values; yp=rf.predict(imp.transform(d.loc[ts,feat].values))
            rec.append({"scenario":sc,"rep":rep,"R2":r2_score(yt,yp),
                        "RMSE":float(mean_squared_error(yt,yp)**0.5),"MBE":float(np.mean(yp-yt)),
                        "MAE":float(mean_absolute_error(yt,yp)),"n_test":len(yt)})
    return rf, imp, pd.DataFrame(rec)

In [6]:
def run_leave_one_in(t, df_raw):
    c=TOWER_CONFIGS[t]; d=df_raw.copy(); tgt=c["target"]
    d.loc[~d[c["ssitc"]].isin([0,1]),tgt]=np.nan
    d.loc[d[tgt].notna() & ~d[tgt].between(PLAUS_LOW,PLAUS_HIGH,inclusive="both"),tgt]=np.nan
    d,BASE,groups=build_features(d,c)
    tr=d.index.year.isin(c["train_yrs"]); base_tr=d.loc[tr,BASE+[tgt]].dropna()
    tix=base_tr.index; ytr=d.loc[tix,tgt].values
    rep_gaps={sc:insert_calendar_gaps(d,tgt,c["test_yrs"],gh) for sc,gh in SCENARIOS.items()}
    sets={"BASE":BASE}
    for g,cols in groups.items(): sets[g]=BASE+[x for x in cols if x not in BASE]
    sets["ALL"]=BASE+[x for g in groups.values() for x in g if x not in BASE]
    out={}
    print(f"\nTower {t} (leave-one-group-in): train={len(tix):,}")
    for name,feat in sets.items():
        feat=list(dict.fromkeys(feat))
        _,_,m=fit_eval(d,tgt,feat,tix,ytr,rep_gaps)
        m["feature_set"]=name; out[name]=m
        print(f"  {name:18s} nfeat={len(feat):3d}  median R2={m['R2'].median():+.3f}")
    return pd.concat(out.values(), ignore_index=True)

In [7]:
def generic_frame(t, df_raw, use_density):
    """Standardized-column frame for pooling. livestock = density or count."""
    c=TOWER_CONFIGS[t]; d=df_raw.copy(); tgt=c["target"]
    d.loc[~d[c["ssitc"]].isin([0,1]),tgt]=np.nan
    d.loc[d[tgt].notna() & ~d[tgt].between(PLAUS_LOW,PLAUS_HIGH,inclusive="both"),tgt]=np.nan
    add_time(d); liv=c["livestock"]; area=c["area_ha"]
    lsu=sum(d[liv[s]].fillna(0)*w for s,w in LSU.items())
    g=pd.DataFrame(index=d.index)
    g["target"]=d[tgt]
    for k in ["sw","ta","vpd","ppfd","ustar","ws","rn","precip","ts","swc","shf","fc"]:
        g[k]=d[c[k]]
    for a in AUX: g[a]=d[a]
    g["lsu"]=(lsu/area) if use_density else lsu
    g["graze"]=(sum(d[liv[s]].fillna(0) for s in LSU)>0).astype(float)
    g["_year"]=d.index.year
    return g, tgt

POOL_FEAT=["sw","ta","vpd","ppfd","ustar","ws","rn","precip","ts","swc","shf","fc"]+AUX+["lsu","graze"]

def run_pooled(df_raw, use_density):
    # train across T2(2018)+T4+T9, evaluate on T4 & T9 test
    frames={t:generic_frame(t,df_raw,use_density) for t in [2,4,9]}
    tr_parts=[]
    for t in [2,4,9]:
        g,_=frames[t]; yrs=TOWER_CONFIGS[t]["train_yrs"]
        sub=g[g["_year"].isin(yrs)][POOL_FEAT+["target"]].dropna()
        tr_parts.append(sub)
    train=pd.concat(tr_parts, ignore_index=True)
    imp=SimpleImputer(strategy="mean"); rf=RandomForestRegressor(n_estimators=500,min_samples_leaf=5,n_jobs=-1,random_state=42)
    rf.fit(imp.fit_transform(train[POOL_FEAT].values), train["target"].values)
    rows=[]
    for t in [4,9]:
        g,_=frames[t]; c=TOWER_CONFIGS[t]
        rep_gaps={sc:insert_calendar_gaps(g,"target",c["test_yrs"],gh) for sc,gh in SCENARIOS.items()}
        for sc,reps in rep_gaps.items():
            for rep,ts in enumerate(reps):
                if len(ts)<5: continue
                yt=g.loc[ts,"target"].values; yp=rf.predict(imp.transform(g.loc[ts,POOL_FEAT].values))
                rows.append({"tower":f"Tower {t}","scenario":sc,"rep":rep,"R2":r2_score(yt,yp),
                             "RMSE":float(mean_squared_error(yt,yp)**0.5),"MBE":float(np.mean(yp-yt))})
    return train.shape[0], pd.DataFrame(rows)

## 4  Run leave-one-group-in (Towers 4 & 9)

In [8]:
loo={}
for t in [4,9]: loo[t]=run_leave_one_in(t, df_raw)


Tower 4 (leave-one-group-in): train=7,285


  BASE               nfeat= 16  median R2=+0.061


  livestock_count    nfeat= 23  median R2=+0.109


  livestock_density  nfeat= 20  median R2=+0.099


  mgmt_pruned        nfeat= 18  median R2=+0.081


...

  wind               nfeat= 18  median R2=-0.019


  moisture           nfeat= 19  median R2=-0.070


  soiltemp_prod      nfeat= 18  median R2=-0.021


  chem               nfeat= 20  median R2=-0.055


  ALL                nfeat= 40  median R2=+0.030


## 5  Pooled (T2+T4+T9) — stocking count vs density

In [9]:
ntr_c, pooled_count = run_pooled(df_raw, use_density=False)
ntr_d, pooled_dens  = run_pooled(df_raw, use_density=True)
print(f"Pooled train rows: count={ntr_c:,}  density={ntr_d:,}")
for lbl,pp in [("POOLED count", pooled_count),("POOLED density", pooled_dens)]:
    print(f"\n{lbl}: median R2 by tower x scenario")
    print(pp.groupby(["tower","scenario"])["R2"].median().unstack("scenario").reindex(columns=list(SCENARIOS)).round(3).to_string())

Pooled train rows: count=12,558  density=12,558

POOLED count: median R2 by tower x scenario
scenario     vs      s      m      l     m1
tower                                      
Tower 4   0.239  0.160  0.212 -0.014 -0.107
Tower 9   0.091  0.015  0.179 -0.107  0.227

POOLED density: median R2 by tower x scenario
scenario     vs      s      m      l    m1
tower                                     
Tower 4   0.239  0.151  0.236  0.026 -0.11
Tower 9   0.287  0.217  0.294  0.207  0.25


## 6  Results — leave-one-group-in tables (Δ vs BASE)

In [10]:
ORDER=["BASE","livestock_count","livestock_density","mgmt_pruned","wind","moisture","soiltemp_prod","chem","ALL"]
for t,m in loo.items():
    tbl=(m.groupby(["feature_set","scenario"])["R2"].median().unstack("scenario")
         .reindex(index=ORDER).reindex(columns=list(SCENARIOS))).round(3)
    print(f"\n=== Tower {t}: median R2 by group (leave-one-in) ===")
    print(tbl.to_string())
    print("\n  Delta vs BASE:")
    print((tbl-tbl.loc["BASE"]).round(3).to_string())
    print(f"  [check] count vs density identical? "
          f"{np.allclose(tbl.loc['livestock_count'].values, tbl.loc['livestock_density'].values, equal_nan=True)}")


=== Tower 4: median R2 by group (leave-one-in) ===
scenario              vs      s      m      l     m1
feature_set                                         
BASE               0.156  0.084  0.111  0.031 -0.121
livestock_count    0.256  0.159  0.186  0.014 -0.072
livestock_density  0.247  0.150  0.178  0.008 -0.078
mgmt_pruned        0.178  0.084  0.114  0.022 -0.098
wind               0.165  0.105  0.139  0.051 -0.070
moisture           0.164  0.090  0.122  0.038 -0.082
soiltemp_prod      0.141  0.071  0.097  0.033 -0.143
chem               0.154  0.083  0.112  0.031 -0.125
ALL                0.262  0.162  0.168  0.024 -0.024

  Delta vs BASE:
scenario              vs      s      m      l     m1
feature_set                                         
BASE               0.000  0.000  0.000  0.000  0.000
livestock_count    0.100  0.075  0.075 -0.017  0.049
livestock_density  0.091  0.066  0.067 -0.023  0.043
mgmt_pruned        0.022  0.000  0.003 -0.009  0.023
wind               0.009  0.0

## 7  Append to benchmarks.csv

In [11]:
bench=RESULTS/"benchmarks.csv"; today=datetime.date.today().isoformat()
ex=pd.read_csv(bench); ex=ex[ex["replication"]!="F-02"]
rows=[]
for t,m in loo.items():
    for r in m.to_dict("records"):
        rows.append({"replication":"F-02","model":"RFm","tower":f"Tower {t}","feature_set":r["feature_set"],
            "scenario":r["scenario"],"rep":r["rep"],"split":"test_2022-2023","R2":round(r["R2"],4),
            "RMSE":round(r["RMSE"],4),"MAE":round(r["MAE"],4),"MBE":round(r["MBE"],4),"n_test":r["n_test"],
            "date":today,"notes":"F02 leave-one-group-in; pruned mgmt; +stocking density (D-29)"})
for lbl,pp in [("POOLED_count",pooled_count),("POOLED_density",pooled_dens)]:
    for r in pp.to_dict("records"):
        rows.append({"replication":"F-02","model":"RFm_pooled","tower":r["tower"],"feature_set":lbl,
            "scenario":r["scenario"],"rep":r["rep"],"split":"pool_T2+4+9_test_2022-2023","R2":round(r["R2"],4),
            "RMSE":round(r["RMSE"],4),"MAE":"","MBE":round(r["MBE"],4),"n_test":"",
            "date":today,"notes":"Pooled T2(2018)+T4+T9; count vs LSU/ha density (D-29)"})
new=pd.DataFrame(rows); comb=pd.concat([ex,new],ignore_index=True); comb.to_csv(bench,index=False)
print(f"Wrote {len(new)} F-02 rows. Total {len(comb)}.")

Wrote 550 F-02 rows. Total 1840.
